In [ ]:
import requests

#авторизация на MOEX ISS через basic-аутентификацию
MOEX_LOGIN = 'orlovapolya0211@gmail.com' #логин от moex.com
MOEX_PASSWORD = 'E!3UStXK#wAwLgh'  #пароль от moex.com

session = requests.Session()
avtoriz_otvet = session.get(
    'https://passport.moex.com/authenticate',
    auth=(MOEX_LOGIN, MOEX_PASSWORD))

if avtoriz_otvet.status_code == 200:
    print('Авторизация успешна')
    print(f'Cookie получен: {"MicexPassportCert" in session.cookies}')
else:
    print(f'Ошибка авторизации: {avtoriz_otvet.status_code}')

Авторизация успешна
Cookie получен: True


In [ ]:
#cоздаём конфигурационный файл parametrs.txt
#параметры берём из статьи Tproger
#там написано - у basicConfig() три основных параметра  - level, filename, format

parametrs_content = """enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
"""
#enabled=true/false  - включение/выключение logging без изменения кода
#level=INFO
#filename=logs/logging_infa - где хранятся наши логи
#Tproger: format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
#убрали funcName потому что у нас нет функций
#%(asctime)s   - дата и время события
#%(levelname)s  - уровень: INFO в нашем случае
#%(message)s   - само сообщение
#%(lineno)d - номер строки


with open('parametrs.txt', 'w') as filik:
    filik.write(parametrs_content)

print('создали parametrs.txt с такими параметрами:')
with open('parametrs.txt') as filik:
    print(filik.read())

создали parametrs.txt с такими параметрами:
enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s



In [ ]:
!mkdir -p logs #создали папку, где будет файл с логами

In [ ]:
import logging

#читаем наши настройки из parametrs.txt
with open('parametrs.txt') as filik:
    lines = filik.readlines()

enabled = lines[0].split('=')[1].strip()
level_str = lines[1].split('=')[1].strip()
filename = lines[2].split('=')[1].strip()
log_format = lines[3].split('=')[1].strip()

if enabled == 'true':

    #сбрасываем старые handlers
    #потому что иначе basicConfig игнорируется в Colab
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    #filename  - из parametrs.txt
    #из Хабра нашли, что записываем сообщения в файл
    #файл будет хранить данные после завершения программы

    #format  - из parametrs.txt
    #Tproger: format с %(asctime)s %(levelname)s %(message)s

    #запись в файл
    #Tproger: 'logging.basicConfig()  - основная функция для настройки logging'
    #уровень INFO  - вывод данных о фрагментах кода, работающих так как ожидается
    logging.basicConfig(level=logging.INFO, filename=filename, format=log_format, filemode='a')
    #filemode='a'  - логи добавляются в конец файла и не стираются при перезапуске

    logging.info('Подключаем logging и собираем данные с MOEX ISS API')
    print(f'Logging работает: уровень={level_str}, файл={filename}')

else:
    print('Logging не работает')

Logging работает: уровень=INFO, файл=logs/logging_infa


In [ ]:
import requests #сначала все запросы писали через requests.get() напрямую, потом переписали через session для логгирования
import pandas as pd

Импортируем две библиотеки:
- **requests**  - для отправки HTTP-запросов к API биржи (для колаба без логгирования нужна была)
- **pandas**  - для работы с таблицами. Превращаем неудобный JSON в удобный DataFrame

## Запрос 1  - Дневные свечи (candles)

**Цель:** получить историю дневных цен Газпрома за период 2018-01-01  - 2025-12-31.

Это главные данные всего проекта. Без них невозможно посчитать нашу таргетную переменную  - изменение цены акции через 24 часа после выхода новости.

Каждая строка в ответе = один торговый день.


In [ ]:
#запрос 1: получаем дневные свечи ГАЗПРОМ с MOEX (первые 500 строк)
response_GAZP = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GAZP/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24})

data_GAZP = response_GAZP.json()

#извлекаем данные из json
columns_GAZP = data_GAZP['candles']['columns']
rows_GAZP = data_GAZP['candles']['data']

df_GAZP = pd.DataFrame(rows_GAZP, columns=columns_GAZP)
df_GAZP['ticker'] = 'GAZP'

print(f'Count of rows: {len(df_GAZP)}')
#считаем все строки  - мало ли их много, а выводит только 500
df_GAZP


Count of rows: 500


,open,close,high,low,value,volume,begin,end,ticker
0,131.03,132.20,132.20,130.63,1.384860e+09,10533920,2018-01-03 00:00:00,2018-01-03 23:59:59,GAZP
1,132.50,135.89,136.20,132.30,4.332522e+09,32096510,2018-01-04 00:00:00,2018-01-04 23:59:59,GAZP
2,135.60,137.12,137.12,135.08,2.725954e+09,19981160,2018-01-05 00:00:00,2018-01-05 23:59:59,GAZP
3,138.00,140.00,141.35,137.60,7.341282e+09,52493430,2018-01-09 00:00:00,2018-01-09 23:59:59,GAZP
4,140.49,143.43,143.43,139.58,7.339464e+09,51773200,2018-01-10 00:00:00,2018-01-10 23:59:59,GAZP
...,...,...,...,...,...,...,...,...,...
495,249.60,250.78,252.22,249.20,7.429337e+09,29618770,2019-12-16 00:00:00,2019-12-16 23:59:59,GAZP
496,251.23,252.05,252.86,250.80,7.974683e+09,31637520,2019-12-17 00:00:00,2019-12-17 23:59:59,GAZP
497,252.00,251.29,252.71,250.85,7.347363e+09,29208640,2019-12-18 00:00:00,2019-12-18 23:59:59,GAZP
498,250.96,251.40,252.30,249.40,9.373045e+09,37390530,2019-12-19 00:00:00,2019-12-19 23:59:59,GAZP


**Результат:**  Count of rows: 500

Таблица содержит 9 колонок:
-  open   - цена акции Газпрома в начале торгового дня (рублей за акцию)
-  close   - цена в конце торгового дня
-  high   - максимальная цена за день
-  low   - минимальная цена за день
-  value   - суммарный оборот за день в рублях (сколько рублей прошло через все сделки)
-  volume   - количество акций сменивших владельца за день
-  begin   - дата и время начала торгового дня
-  end   - дата и время конца торгового дня
-  ticker   - тикер бумаги (добавили сами, чтобы потом объединить все 7 компаний)


Также из документации (стр. 11) узнали что сервер возвращает данные порциями. Чтобы выгрузить полный объём данных, нужно последовательно делать запросы, увеличивая параметр start на значение pagesize, пока не получим пустой блок данных.



In [ ]:
logging.info('Запрос 1: проверяем есть ли данные после 500 строки (start=500)')
#проверяем есть ли данные ниже 500 строки
response_GAZP = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GAZP/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 500
      })

data_GAZP = response_GAZP.json()
print(f"Count of rows after 500 rows: {len(data_GAZP['candles']['data'])}")
#строк много  - выводим оставшиеся


Count of rows after 500 rows: 500


**Результат:**  Count of rows after 500 rows: 500

HOOORRAY!!))

Ну вот видите?)

Данные не закончились на 500-й строке. Получили ещё 500 строк  - это оставшиеся торговые дни, которые не влезли в первый запрос.

Теперь будем проверять, пока нам не выведет 0 значений.

In [ ]:
columns_GAZPnew = data_GAZP['candles']['columns']
rows_GAZPnew = data_GAZP['candles']['data']

df_GAZPnew = pd.DataFrame(rows_GAZPnew, columns=columns_GAZPnew)
df_GAZPnew['ticker'] = 'GAZP'

print(f'Count of rows after 500 rows: {len(df_GAZPnew)}')

#проверяем есть ли данные ниже 1000 строки
logging.info('Запрос 1: проверяем есть ли данные после 1000 строки (start=1000)')
response_GAZP2 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GAZP/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1000
      })

data_GAZP2 = response_GAZP2.json()
columns_GAZPnew2 = data_GAZP2['candles']['columns']
rows_GAZPnew2 = data_GAZP2['candles']['data']

df_GAZPnew2 = pd.DataFrame(rows_GAZPnew2, columns=columns_GAZPnew2)
df_GAZPnew2['ticker'] = 'GAZP'

print(f'Count of rows after 1000 rows: {len(df_GAZPnew2)}')

#проверяем есть ли данные ниже 1500 строки
logging.info('Запрос 1: проверяем есть ли данные после 1500 строки (start=1500)')
response_GAZP3 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GAZP/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1500
      })

data_GAZP3 = response_GAZP3.json()
columns_GAZPnew3 = data_GAZP3['candles']['columns']
rows_GAZPnew3 = data_GAZP3['candles']['data']

df_GAZPnew3 = pd.DataFrame(rows_GAZPnew3, columns=columns_GAZPnew3)
df_GAZPnew3['ticker'] = 'GAZP'

print(f'Count of rows after 1500 rows: {len(df_GAZPnew3)}')



Count of rows after 500 rows: 500
Count of rows after 1000 rows: 500
Count of rows after 1500 rows: 500


In [ ]:
#проверяем есть ли данные ниже 2000 строки

logging.info('Запрос 1: проверяем есть ли данные после 2000 строки (start=2000)')
response_GAZP4 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GAZP/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2000
      })

data_GAZP4 = response_GAZP4.json()
columns_GAZPnew4 = data_GAZP4['candles']['columns']
rows_GAZPnew4 = data_GAZP4['candles']['data']

df_GAZPnew4 = pd.DataFrame(rows_GAZPnew4, columns=columns_GAZPnew4)
df_GAZPnew4['ticker'] = 'GAZP'

print(f'Count of rows after 2000 rows: {len(df_GAZPnew4)}')

Count of rows after 2000 rows: 73


In [ ]:
#проверяем есть ли данные ниже 2073 строки
logging.info('Запрос 1: проверяем есть ли данные после 2073 строки (start=2073)')
response_GAZP5 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GAZP/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2073
      })

data_GAZP5 = response_GAZP5.json()
columns_GAZPnew5 = data_GAZP5['candles']['columns']
rows_GAZPnew5 = data_GAZP5['candles']['data']

df_GAZPnew5 = pd.DataFrame(rows_GAZPnew5, columns=columns_GAZPnew5)
df_GAZPnew5['ticker'] = 'GAZP'

print(f'Count of rows after 2073 rows: {len(df_GAZPnew5)}')

Count of rows after 2073 rows: 0


In [ ]:
#больше данных нет, поэтому обьединяем датафреймы в один
df_GAZPcandles = pd.concat([df_GAZP, df_GAZPnew, df_GAZPnew2, df_GAZPnew3, df_GAZPnew4], ignore_index=True)
print(f'Total rows: {len(df_GAZPcandles)}')

logging.info(f'Запрос 1: итого после склейки всех частей пяти датафреймов  - {len(df_GAZPcandles)} строк')
df_GAZPcandles

Total rows: 2073


,open,close,high,low,value,volume,begin,end,ticker
0,131.03,132.20,132.20,130.63,1.384860e+09,10533920,2018-01-03 00:00:00,2018-01-03 23:59:59,GAZP
1,132.50,135.89,136.20,132.30,4.332522e+09,32096510,2018-01-04 00:00:00,2018-01-04 23:59:59,GAZP
2,135.60,137.12,137.12,135.08,2.725954e+09,19981160,2018-01-05 00:00:00,2018-01-05 23:59:59,GAZP
3,138.00,140.00,141.35,137.60,7.341282e+09,52493430,2018-01-09 00:00:00,2018-01-09 23:59:59,GAZP
4,140.49,143.43,143.43,139.58,7.339464e+09,51773200,2018-01-10 00:00:00,2018-01-10 23:59:59,GAZP
...,...,...,...,...,...,...,...,...,...
2068,124.15,127.37,127.44,124.01,7.571628e+09,60013930,2025-12-26 00:00:00,2025-12-26 23:59:58,GAZP
2069,127.50,127.65,127.95,127.37,1.158205e+09,9071690,2025-12-27 00:00:00,2025-12-27 23:59:57,GAZP
2070,127.88,127.83,128.07,126.80,1.051423e+09,8245610,2025-12-28 00:00:00,2025-12-28 23:59:57,GAZP
2071,127.83,125.29,129.71,124.75,1.355392e+10,106661060,2025-12-29 00:00:00,2025-12-29 23:59:59,GAZP


Hooray!

Пустой ответ подтверждает, что мы выгрузили абсолютно все данные. После 2073 строки данных нет. Значит итоговый датафрейм  df_GAZPcandles  содержит полную историю дневных котировок Газпрома за весь нужный период без пропусков.

**Результат:**
   
Total rows: 2073
   

Склеили пять кусков через  pd.concat  с параметром  ignore_index=True   - пересчитаем индексы строк заново от 0, иначе в итоговой таблице было бы пять наборов индексов, что вызвало бы у нас путаницу и отсутствие hoorray(.

Итого получили **2073 строк**  - это все торговые дни Газпрома за период с 01.01.2018 по 31.12.2025.



## Запрос 2  - Информация о бумаге (security info)

**Цель:** получить полную информацию по акциям Газпрома: официальное название компании, международный код бумаги (ISIN), номинальная стоимость акции и уровень листинга.

Эти данные нужны нам для двух вещей:
1. Красиво описать датасет в README  - чтобы было понятно с какими именно бумагами работаем
2. Добавить уровень листинга как признак в EDA  - бумаги первого уровня листинга торгуются активнее и имеют более строгие требования к раскрытию информации

In [ ]:
#запрос 2: получаем статические характеристики бумаги ГАЗПРОМ
#нам возвращается: полное название, ISIN, номинал, уровень листинга, тип бумаги
response_GAZP_infa = session.get(
    'https://iss.moex.com/iss/securities/GAZP.json')

data_GAZP_infa = response_GAZP_infa.json()

#раздел 'description' содержит характеристики в виде строк name/value
columns_GAZP_infa = data_GAZP_infa['description']['columns']
rows_GAZP_infa = data_GAZP_infa['description']['data']

df_GAZP_infa = pd.DataFrame(rows_GAZP_infa, columns=columns_GAZP_infa)
df_GAZP_infa['ticker'] = 'GAZP'

print(f'Count of rows: {len(df_GAZP_infa)}')
df_GAZP_infa

Count of rows: 27


,name,title,value,type,sort_order,is_hidden,precision,ticker
0,SECID,Код ценной бумаги,GAZP,string,1,0,NaN,GAZP
1,ISSUENAME,Наименование ценной бумаги,Акции обыкновенные,string,2,0,NaN,GAZP
2,NAME,Полное наименование,"""Газпром"" (ПАО) ао",string,3,0,NaN,GAZP
3,SHORTNAME,Краткое наименование,ГАЗПРОМ ао,string,4,0,NaN,GAZP
4,ISIN,ISIN код,RU0007661625,string,5,0,NaN,GAZP
5,REGNUMBER,Номер государственной регистрации,1-02-00028-A,string,6,0,NaN,GAZP
6,ISSUESIZE,Объем выпуска,23673512900,number,7,0,0.0,GAZP
7,FACEVALUE,Номинальная стоимость,5,number,8,0,2.0,GAZP
8,FACEUNIT,Валюта номинала,SUR,string,9,0,NaN,GAZP
9,ISSUEDATE,Дата начала торгов,2006-01-23,date,10,0,NaN,GAZP


**Результат:**  Count of rows: 27

Получили 27 строк.

Каждая строка это одна характеристика бумаги.

Каждая характеристика на отдельной строке.

Это неудобно для анализа, поэтому в следующей ячейке мы преобразуем это формат, где одна строка на компанию, каждая характеристика отдельная колонка.

In [ ]:
#преобразуем в удобный формат: одна строка  - один тикер
#разворачиваем колонку 'name' в отдельные колонки через pivot
df_GAZP_infa_best = df_GAZP_infa.set_index('name')['value'].to_frame().T
df_GAZP_infa_best['ticker'] = 'GAZP'
df_GAZP_infa_best = df_GAZP_infa_best.reset_index()

print(f'Полное название: {df_GAZP_infa_best["NAME"].values[0]}')
print(f'ISIN: {df_GAZP_infa_best["ISIN"].values[0]}')
print(f'Уровень листинга: {df_GAZP_infa_best["LISTLEVEL"].values[0]}')

logging.info(f'Запрос 2: ISIN={df_GAZP_infa_best["ISIN"].values[0]}, листинг={df_GAZP_infa_best["LISTLEVEL"].values[0]}')

df_GAZP_infa_best

Полное название: "Газпром" (ПАО) ао
ISIN: RU0007661625
Уровень листинга: 1


name,index,SECID,ISSUENAME,NAME,SHORTNAME,ISIN,REGNUMBER,ISSUESIZE,FACEVALUE,FACEUNIT,...,MORNINGSESSION,EVENINGSESSION,WEEKENDSESSION,REGISTRY_DATE,TYPENAME,GROUP,TYPE,GROUPNAME,EMITTER_ID,ticker
0,value,GAZP,Акции обыкновенные,"""Газпром"" (ПАО) ао",ГАЗПРОМ ао,RU0007661625,1-02-00028-A,23673512900,5,SUR,...,1,1,1,1998-12-30,Акция обыкновенная,stock_shares,common_share,Акции,711,GAZP


**Результат:**
   
Полное название: Газпром ПАО ао
ISIN: RU0007661625
Уровень листинга: 1
   

Теперь это одна строка с колонками NAME, ISIN, LISTLEVEL и тд.

Что значат выводы:
- **NAME = «Газпром ПАО ао»**  - официальное юридическое название компании.
- **ISIN = RU0007661625**  - международный идентификационный номер ценной бумаги.
- **LISTLEVEL = 1**  - первый уровень листинга на Мосбирже. Это значит, что Газпром прошёл самые строгие требования биржи по прозрачности и раскрытию информации.

**Итог запроса 2:** получили официальную информацию по бумаге.

Hooray!))

## Запрос 3  - История дивидендов (dividends)

**Цель:** получить все даты дивидендных выплат Газпрома за период проекта.

Зачем нам это нужно: когда компания выплачивает дивиденды, цена акции резко падает примерно на размер дивиденда.

Если мы не пометим эти дни, то можем ошибочно решить в EDA, что резкое падение цены вызвано негативной новостью  - хотя на самом деле это просто плановая выплата. Это могло бы нам помешать.

In [ ]:
logging.info('Запрос 3: загружаем историю дивидендов GAZP')
#запрос 3: получаем историю дивидендных выплат ГАЗПРОМ
#дивиденды влияют на котировки примерно на размер дивиденда
GAZP_diva = session.get(
    'https://iss.moex.com/iss/securities/GAZP/dividends.json')

GAZP_diva_infa = GAZP_diva.json()

#извлекаем данные из json
columns_GAZP_diva = GAZP_diva_infa['dividends']['columns']
rows_GAZP_diva = GAZP_diva_infa['dividends']['data']

df_GAZP_diva = pd.DataFrame(rows_GAZP_diva, columns=columns_GAZP_diva)
df_GAZP_diva['ticker'] = 'GAZP'

print(f'Count of rows: {len(df_GAZP_diva)}')

logging.info(f'Запрос 3: получено {len(df_GAZP_diva)} дивидендных выплат')
df_GAZP_diva

Count of rows: 10


,secid,isin,registryclosedate,value,currencyid,ticker
0,GAZP,RU0007661625,2014-07-17,7.20,RUB,GAZP
1,GAZP,RU0007661625,2015-07-16,7.20,RUB,GAZP
2,GAZP,RU0007661625,2016-07-20,7.89,RUB,GAZP
3,GAZP,RU0007661625,2017-07-20,8.04,RUB,GAZP
4,GAZP,RU0007661625,2018-07-19,8.04,RUB,GAZP
5,GAZP,RU0007661625,2019-07-18,16.61,RUB,GAZP
6,GAZP,RU0007661625,2020-07-16,15.24,RUB,GAZP
7,GAZP,RU0007661625,2021-07-15,12.55,RUB,GAZP
8,GAZP,RU0007661625,2022-07-20,52.53,RUB,GAZP
9,GAZP,RU0007661625,2022-10-11,51.03,RUB,GAZP


**Результат:**  Count of rows: 10

API вернул 10 выплат дивидентов за все годы существования на бирже. Таблица содержит колонки:

- **registryclosedate**  - дата закрытия реестра.
- **value**  - размер дивиденда на одну акцию в рублях.

Нам нужны только выплаты за наш период 2018–2025, поэтому сейчас будет фильтровать.

In [ ]:
#фильтруем дивиденды по диапазону дат проекта 2018-01-01 - 2025-12-31
df_GAZP_diva['registryclosedate'] = pd.to_datetime(df_GAZP_diva['registryclosedate'])
df_GAZP_diva_srok = df_GAZP_diva[
    (df_GAZP_diva['registryclosedate'] >= '2018-01-01') &
    (df_GAZP_diva['registryclosedate'] <= '2025-12-31')].reset_index()

print(f'Все дивиденты за 2018-01-01 - 2025-12-31: {len(df_GAZP_diva_srok)}')

logging.info(f'Запрос 2: ISIN={df_GAZP_infa_best["ISIN"].values[0]}, листинг={df_GAZP_infa_best["LISTLEVEL"].values[0]}')
df_GAZP_diva_srok

Все дивиденты за 2018-01-01 - 2025-12-31: 6


,index,secid,isin,registryclosedate,value,currencyid,ticker
0,4,GAZP,RU0007661625,2018-07-19,8.04,RUB,GAZP
1,5,GAZP,RU0007661625,2019-07-18,16.61,RUB,GAZP
2,6,GAZP,RU0007661625,2020-07-16,15.24,RUB,GAZP
3,7,GAZP,RU0007661625,2021-07-15,12.55,RUB,GAZP
4,8,GAZP,RU0007661625,2022-07-20,52.53,RUB,GAZP
5,9,GAZP,RU0007661625,2022-10-11,51.03,RUB,GAZP


**Результат:**  Все дивиденты за 2018-01-01 - 2025-12-31: 6


Почему конвертировали дату через  pd.to_datetime : без этого Python сравнивал бы строки а не даты. Строковое сравнение работает по алфавиту и даёт неверные результаты для дат в формате YYYY-MM-DD.

**Итог запроса 3:** знаем точные даты дивидендных выплат. В EDA пометим эти дни флагом  is_dividend_day=1 .

## Запрос 4  - Текущие рыночные данные (market data)

Запрос 4 получает текущие торговые данные по Газпрому  - последнюю цену, объём торгов за день, количество сделок и капитализацию компании. Всё это показывает насколько активно торгуется акция прямо сейчас.


Эти данные характеризуют **ликвидность** бумаги  - насколько активно она торгуется. В EDA мы будем использовать их чтобы показать: более ликвидные бумаги быстрее и точнее реагируют на новости, потому что больше участников рынка мгновенно перекладываются после публикации статьи.

In [ ]:
logging.info('Запрос 4: загружаем текущие рыночные данные GAZP')
#запрос 4: получаем текущие торговые данные ГАЗПРОМ
response_GAZP_markdata = session.get(
    'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GAZP.json')

data_GAZP_markdata = response_GAZP_markdata.json()


#раздел 'marketdata' содержит торговые данные текущей/последней сессии
columns_GAZP_markdata = data_GAZP_markdata['marketdata']['columns']
rows_GAZP_markdata = data_GAZP_markdata['marketdata']['data']

df_GAZP_markdata = pd.DataFrame(rows_GAZP_markdata, columns=columns_GAZP_markdata)
df_GAZP_markdata['ticker'] = 'GAZP'

print(f'Count of rows: {len(df_GAZP_markdata)}')

logging.info(f'Запрос 4: получено {len(df_GAZP_markdata)} строк market data')
df_GAZP_markdata

Count of rows: 1


,SECID,BOARDID,BID,BIDDEPTH,OFFER,OFFERDEPTH,SPREAD,BIDDEPTHT,OFFERDEPTHT,OPEN,...,SYSTIME,CLOSINGAUCTIONPRICE,CLOSINGAUCTIONVOLUME,ISSUECAPITALIZATION,ISSUECAPITALIZATION_UPDATETIME,ETFSETTLECURRENCY,VALTODAY_RUR,TRADINGSESSION,TRENDISSUECAPITALIZATION,ticker
0,GAZP,TQBR,130.76,None,130.77,None,0.01,509599,1009571,129.88,...,2026-03-19 20:44:06,130.71,46930,3098389368352,20:43:00,None,14098330832,2,26514334448,GAZP


**Результат:**  Count of rows: 1

Одна строка  - потому что это текущие торговые данные. Но колонок очень много. API возвращает абсолютно всё что знает о торговле этой бумагой прямо сейчас.

Большинство колонок нам не нужны  - это технические поля биржи вроде кодов режимов торгов, флагов состояния и прочего. В следующей ячейке оставим только то что важно для нашего анализа.

In [ ]:
#оставляем только нужные колонки из market data
#используем фильтрацию через список  - так код не упадёт если какой-то колонки нет в ответе
markdata_cols = ['ticker', 'LAST', 'LASTTOPREVPRICE', 'BID', 'OFFER',
            'OPEN', 'HIGH', 'LOW', 'WAPRICE', 'NUMTRADES',
            'VOLTODAY', 'VALTODAY', 'ISSUECAPITALIZATION', 'UPDATETIME']
markdata_cols = [col for col in markdata_cols if col in df_GAZP_markdata.columns]

df_GAZP_markdata_best = df_GAZP_markdata[markdata_cols].copy()

#выводим ключевые показатели
print(f'Последняя цена GAZP: {df_GAZP_markdata_best["LAST"].values[0]}')
print(f'Изменение к пред. закрытию: {df_GAZP_markdata_best["LASTTOPREVPRICE"].values[0]}')
print(f'Капитализация: {df_GAZP_markdata_best["ISSUECAPITALIZATION"].values[0]}')

logging.info(f'Запрос 4: последняя цена GAZP={df_GAZP_markdata_best["LAST"].values[0]}')

df_GAZP_markdata_best

Последняя цена GAZP: 130.78
Изменение к пред. закрытию: 0.79
Капитализация: 3098389368352


,ticker,LAST,LASTTOPREVPRICE,BID,OFFER,OPEN,HIGH,LOW,WAPRICE,NUMTRADES,VOLTODAY,VALTODAY,ISSUECAPITALIZATION,UPDATETIME
0,GAZP,130.78,0.79,130.76,130.77,129.88,132.14,129.16,130.97,149046,107648710,14098330832,3098389368352,20:29:05


**Результат:** (тут данные могут изменяться при каждом запуске)
   
Последняя цена GAZP: 131
Изменение к пред. закрытию: 0.96
Капитализация: 3100283249384
   

Что означают колонки которые мы оставили:
- **LAST**  - последняя цена сделки (рублей за акцию)
- **LASTTOPREVPRICE**  - изменение в % к цене закрытия предыдущего дня
- **BID / OFFER**  - лучшая цена покупки и продажи прямо сейчас. Разница между ними называется спредом  - чем он уже, тем ликвиднее бумага
- **WAPRICE**  - средневзвешенная цена за весь день (учитывает объём каждой сделки)
- **NUMTRADES**  - количество сделок за торговый день. У Газпрома это десятки тысяч  - одна из самых торгуемых бумаг на бирже
- **VOLTODAY / VALTODAY**  - объём в акциях и оборот в рублях за сегодня
- **ISSUECAPITALIZATION**  - рыночная капитализация в рублях.

**Итог запроса 4:** получили характеристику ликвидности Газпрома на текущий момент.

## Запрос 5  - Индексная принадлежность (indices)

**Цель:** узнать в какие биржевые индексы входит Газпром.

Индексные бумаги  - особая категория. Когда акция входит в IMOEX (Индекс МосБиржи), за ней автоматически следят индексные фонды (ETF) и управляющие компании. Это означает что при выходе важной новости реагируют не только частные инвесторы, но и алгоритмы фондов.

В EDA мы сравним: одинаково ли сильно реагируют на новости индексные бумаги и менее крупные компании из нашего списка.

In [ ]:
logging.info('Запрос 5: загружаем индексы GAZP')
#запрос 5: получаем список биржевых индексов в которые входит ГАЗПРОМ
response_GAZP_idishka = session.get(
    'https://iss.moex.com/iss/securities/GAZP/indices.json')

data_GAZP_idishka = response_GAZP_idishka.json()

#извлекаем данные из json
columns_GAZP_idishka = data_GAZP_idishka['indices']['columns']
rows_GAZP_idishka = data_GAZP_idishka['indices']['data']

df_GAZP_idishka = pd.DataFrame(rows_GAZP_idishka, columns=columns_GAZP_idishka)
df_GAZP_idishka['ticker'] = 'GAZP'

print(f'Count of rows: {len(df_GAZP_idishka)}')

logging.info(f'Запрос 5: GAZP входит в {len(df_GAZP_idishka)} индексов')

df_GAZP_idishka

Count of rows: 19


,SECID,SHORTNAME,FROM,TILL,CURRENTVALUE,LASTCHANGEPRC,LASTCHANGE,ticker
0,EPSI,Субиндекс акций,2014-09-16,2026-03-18,1665.38,0.00,-0.02,GAZP
1,HCAP,HCAP,2024-06-21,2024-09-26,907.67,-0.07,-0.66,GAZP
2,ICLIMATE,Индекс МосБиржи Климатический,2025-07-01,2026-03-18,1030.83,0.23,2.39,GAZP
3,IMOEXCNY,Индекс МосБиржи в юанях,2024-09-24,2026-03-18,1060.38,-2.49,-27.11,GAZP
4,IMOEXW,IMOEXW,2023-01-16,2026-03-18,2881.58,-0.15,-4.34,GAZP
5,MESG,Индекс МосБиржи-RAEX ESG,2023-02-27,2023-12-21,984.75,-0.27,-2.69,GAZP
6,MOEXBC,Индекс голубых фишек,2010-01-14,2026-03-19,19057.56,-0.03,-5.43,GAZP
7,MRRT,Индекс MRRT,2020-09-21,2026-03-18,2335.80,0.05,1.15,GAZP
8,MRSV,Индекс MRSV,2020-09-21,2026-03-18,2173.86,-0.22,-4.80,GAZP
9,MRSVR,Индекс МосБиржи-РСПП MRSV RU Co,2021-03-25,2026-03-18,2237.45,-0.22,-4.93,GAZP


**Результат:**  Count of rows: 19


Важные колонки:
- **SECID**  - код индекса
- **FROM / TILL**  - период с которого по который бумага входит в индекс
- **CURRENTVALUE**  - текущее значение самого индекса

In [ ]:
#проверяем входит ли GAZP в IMOEX (главный российский фондовый индекс)
if len(df_GAZP_idishka) > 0:
    imoex_check = df_GAZP_idishka[df_GAZP_idishka['SECID'] == 'IMOEX']
    print(f'GAZP входит в IMOEX: {len(imoex_check) > 0}')
else:
    print('GAZP не входит ни в один индекс MOEX')
logging.info(f'Запрос 5 завершили  - все данные по GAZP собраны')


GAZP входит в IMOEX: True


**Результат:**
   
GAZP входит в IMOEX: True
   

Газпром входит в IMOEX  - это подтверждает что он является влиятельной компанией российского рынка.

**Итог запроса 5:** Газпром входит в IMOEX и другие индексы Мосбиржи. В EDA это будет важным признаком при анализе силы реакции на новости.



